<a href="https://colab.research.google.com/github/Yiozolm/ConverseNet/blob/main/kernel-bound/tools/colab_kernel_bound/kernel_bound_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Converse2D v3.0.0: `scale1_forward` bound check on a Colab GPU

Repeats the RTX 5060 Ti Nsight Compute measurement (B4/C128/100x100, shared prior,
kernel `[1,128,3,3]`, full-spectrum training; DRAM 88.3% / 395 GB/s, SM 31.2%)
on the Colab GPU. Choose **Runtime > Change runtime type > A100 / L4 / H100** first.

The branch `colab/kernel-bound` is `v3.0.0` plus `tools/colab_kernel_bound/` only;
`run.py` checks that `Converse2D/`, `models/` and `test/` match `v3.0.0` byte for byte.

In [1]:
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

Sat Oct  3 10:57:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%cd /content
!rm -rf ConverseNet
!git clone --branch colab/kernel-bound --single-branch https://github.com/Yiozolm/ConverseNet.git
%cd /content/ConverseNet
!git log --oneline -3
!pip -q install ninja

/content
Cloning into 'ConverseNet'...
remote: Enumerating objects: 1900, done.
remote: Counting objects: 100% (1227/1227), done.
remote: Compressing objects: 100% (817/817), done.
remote: Total 1900 (delta 435), reused 1162 (delta 396), pack-reused 673 (from 1)
Receiving objects: 100% (1900/1900), 201.27 MiB | 18.03 MiB/s, done.
Resolving deltas: 100% (602/602), done.
/content/ConverseNet
b8030de (HEAD -> colab/kernel-bound, origin/colab/kernel-bound) test: add Colab harness for v3.0.0 scale1_forward bound check
0d63621 (tag: v3.0.0) style: apply clangd formatting and ignore VS Code settings
f6250a8 Remove experiment documentation from the release tree
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 16.2 MB/s eta 0:00:00


Nsight Compute is optional. If `ncu` is missing, this tries the NVIDIA apt repo.
If the host forbids GPU counters (`ERR_NVGPUCTRPERM`), the run records that and
keeps the counter-free timing result.

In [3]:
import glob, shutil, subprocess
ncu = shutil.which('ncu') or (sorted(glob.glob('/usr/local/cuda*/bin/ncu') + glob.glob('/usr/local/cuda*/nsight-compute*/ncu') + glob.glob('/opt/nvidia/nsight-compute/*/ncu')) or [None])[-1]
if ncu is None:
    pkgs = subprocess.run("apt-cache search --names-only '^nsight-compute-[0-9]' | cut -d' ' -f1 | sort -V | tail -1", shell=True, capture_output=True, text=True).stdout.strip()
    print('installing', pkgs or '(none found)')
    if pkgs:
        subprocess.run(f'apt-get -qq update && apt-get -qq install -y {pkgs}', shell=True)
    ncu = (sorted(glob.glob('/opt/nvidia/nsight-compute/*/ncu') + glob.glob('/usr/local/cuda*/nsight-compute*/ncu')) or [None])[-1]
print('ncu:', ncu)
if ncu:
    import os; os.environ['PATH'] = os.path.dirname(ncu) + ':' + os.environ['PATH']
    !ncu --version

ncu: /usr/local/cuda/bin/ncu
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.3.1.0 (build 36398880) (public-release)


In [4]:
import time
OUT = f"/content/results/{torch.cuda.get_device_name(0).replace(' ', '_')}-{time.strftime('%Y%m%dT%H%M%S')}"
!python -u tools/colab_kernel_bound/run.py --output {OUT}

Building extension (first build takes several minutes)...
Probing copy bandwidth and FP32 GEMM...
Kernel timing via torch.profiler...
NCU capture A: scale1_forward, --set full...
NCU capture B: SpeedOfLight for every kernel...
# scale1_forward bound check: NVIDIA A100-SXM4-40GB (sm_80)

- torch 2.11.0+cu130 / CUDA 13.0; production matches v3.0.0: True
- copy probe: 1378.8 GB/s; FP32 GEMM probe: 19.14 TFLOP/s (TF32 off)

| source | DRAM % | DRAM GB/s | SM % | L2 % | duration us | classification |
|---|---|---|---|---|---|---|
| RTX 5060 Ti (recorded) | 88.34 | 395.4 | 31.23 | - | - | memory-bound (DRAM) |
| NVIDIA A100-SXM4-40GB NCU | 80.35 | 1248.9 | 72.39 | 68.33 | 91.1 | memory-bound (DRAM 80.3%) |

Timing-only (no counters): scale1_forward median 101.3 us, >= 959.9 GB/s = 69.6% of the copy probe.

## All kernels in one training call (SpeedOfLight)

| kernel | duration us | DRAM % | L2 % | SM % | classification |
|---|---|---|---|---|---|
| `void converse2d::psf_pad_roll_kernel<int>(

In [5]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/summary.md').read()))

# scale1_forward bound check: NVIDIA A100-SXM4-40GB (sm_80)

- torch 2.11.0+cu130 / CUDA 13.0; production matches v3.0.0: True
- copy probe: 1378.8 GB/s; FP32 GEMM probe: 19.14 TFLOP/s (TF32 off)

| source | DRAM % | DRAM GB/s | SM % | L2 % | duration us | classification |
|---|---|---|---|---|---|---|
| RTX 5060 Ti (recorded) | 88.34 | 395.4 | 31.23 | - | - | memory-bound (DRAM) |
| NVIDIA A100-SXM4-40GB NCU | 80.35 | 1248.9 | 72.39 | 68.33 | 91.1 | memory-bound (DRAM 80.3%) |

Timing-only (no counters): scale1_forward median 101.3 us, >= 959.9 GB/s = 69.6% of the copy probe.

## All kernels in one training call (SpeedOfLight)

| kernel | duration us | DRAM % | L2 % | SM % | classification |
|---|---|---|---|---|---|
| `void converse2d::psf_pad_roll_kernel<int>(const float *, float *, T1, ` | 14.3 | 0.0 | 20.6 | 60.8 | compute-bound (SM 60.8%) |
| `void at::unrolled_elementwise_kernel<at::direct_copy_kernel_cuda(at::T` | 12.5 | 26.7 | 59.0 | 40.0 | latency/occupancy-limited (no unit >= 60% of peak; top L2 59.0%) |
| `void regular_fft<100, EPT<10>, 16, 7, 6, 0, 2, 1, unsigned int, float,` | 18.2 | 39.1 | 60.2 | 30.7 | memory-bound (L2 60.2%) |
| `void regular_fft<100, EPT<10>, 16, 9, 6, 0, 2, 0, unsigned int, float,` | 19.4 | 34.1 | 55.8 | 28.7 | latency/occupancy-limited (no unit >= 60% of peak; top L2 55.8%) |
| `void at::unrolled_elementwise_kernel<at::direct_copy_kernel_cuda(at::T` | 41.8 | 67.9 | 68.0 | 46.4 | memory-bound (L2 68.0%) |
| `void regular_fft<100, EPT<10>, 16, 7, 6, 0, 2, 1, unsigned int, float,` | 58.0 | 74.4 | 75.2 | 38.6 | memory-bound (L2 75.2%) |
| `void regular_fft<100, EPT<10>, 16, 9, 6, 0, 2, 0, unsigned int, float,` | 57.0 | 73.5 | 71.6 | 38.5 | memory-bound (DRAM 73.5%) |
| `void at::vectorized_elementwise_kernel<4, at::CUDAFunctorOnSelf_add<fl` | 3.7 | 0.1 | 0.5 | 0.0 | latency/occupancy-limited (no unit >= 60% of peak; top L1 31.8%) |
| `void at::vectorized_elementwise_kernel<4, at::sigmoid_kernel_cuda(at::` | 4.3 | 0.2 | 0.6 | 0.0 | latency/occupancy-limited (no unit >= 60% of peak; top L1 23.8%) |
| `void at::vectorized_elementwise_kernel<4, at::CUDAFunctorOnSelf_add<fl` | 3.6 | 0.1 | 0.5 | 0.0 | latency/occupancy-limited (no unit >= 60% of peak; top L1 32.3%) |
| `void converse2d::scale1_forward<float>(const c10::complex<T1> *, const` | 89.8 | 81.0 | 68.6 | 73.4 | memory-bound (DRAM 81.0%) |
| `void regular_fft<100, EPT<10>, 16, 7, 6, 0, 2, 1, unsigned int, float,` | 57.6 | 75.0 | 75.1 | 38.8 | memory-bound (L2 75.1%) |
| `void regular_fft<100, EPT<10>, 16, 9, 6, 0, 2, 0, unsigned int, float,` | 57.3 | 73.0 | 70.5 | 39.4 | memory-bound (DRAM 73.0%) |
| `void at::vectorized_elementwise_kernel<2, at::AUnaryFunctor<c10::compl` | 53.2 | 78.5 | 67.4 | 5.9 | memory-bound (DRAM 78.5%) |
| `void at::vectorized_elementwise_kernel<4, at::FillFunctor<float>, std:` | 26.7 | 56.5 | 71.0 | 10.5 | memory-bound (L2 71.0%) |
| `void at::elementwise_kernel<128, 2, void at::gpu_kernel_impl_nocast<at` | 70.9 | 82.8 | 42.7 | 13.7 | memory-bound (DRAM 82.8%) |
| `void regular_fft<100, EPT<10>, 16, 7, 6, 0, 2, 1, unsigned int, float,` | 57.9 | 74.5 | 75.1 | 38.8 | memory-bound (L2 75.1%) |
| `void regular_fft<100, EPT<10>, 16, 9, 6, 0, 2, 0, unsigned int, float,` | 57.1 | 73.3 | 71.3 | 39.1 | memory-bound (DRAM 73.3%) |
| `void at::vectorized_elementwise_kernel<2, at::AUnaryFunctor<c10::compl` | 55.2 | 75.5 | 66.4 | 5.9 | memory-bound (DRAM 75.5%) |
| `void converse2d::scale1_batch_prepare<float>(const c10::complex<T1> *,` | 163.2 | 85.1 | 68.5 | 56.5 | memory-bound (DRAM 85.1%) |
| `void at::reduce_kernel<512, 1, at::ReduceOp<float, at::func_wrapper_t<` | 37.3 | 77.3 | 58.3 | 37.2 | memory-bound (DRAM 77.3%) |
| `void at::reduce_kernel<512, 1, at::ReduceOp<float, at::func_wrapper_t<` | 11.7 | 28.6 | 20.6 | 10.2 | latency/occupancy-limited (no unit >= 60% of peak; top DRAM 28.6%) |
| `void converse2d::scale1_batch_kernel<float, 4>(const c10::complex<T1> ` | 139.6 | 86.0 | 67.6 | 31.3 | memory-bound (DRAM 86.0%) |
| `void at::vectorized_elementwise_kernel<4, at::sigmoid_backward_kernel_` | 3.8 | 0.1 | 0.7 | 0.0 | latency/occupancy-limited (no unit >= 60% of peak; top L1 30.2%) |
| `void regular_fft<100, EPT<10>, 16, 7, 6, 0, 2, 1, unsigned int, float,` | 57.8 | 74.7 | 74.6 | 39.1 | memory-bound (DRAM 74.7%) |
| `void regular_fft<100, EPT<10>, 16, 9, 6, 0, 2, 0, unsigned int, float,` | 57.1 | 73.4 | 72.0 | 39.4 | memory-bound (DRAM 73.4%) |
| `void regular_fft<100, EPT<10>, 16, 7, 6, 0, 2, 1, unsigned int, float,` | 18.3 | 39.1 | 59.8 | 31.1 | latency/occupancy-limited (no unit >= 60% of peak; top L2 59.8%) |
| `void regular_fft<100, EPT<10>, 16, 9, 6, 0, 2, 0, unsigned int, float,` | 19.5 | 34.0 | 56.0 | 28.3 | latency/occupancy-limited (no unit >= 60% of peak; top L2 56.0%) |
| `void converse2d::psf_pad_roll_backward_kernel<int>(const float *, floa` | 4.2 | 0.7 | 1.0 | 0.3 | latency/occupancy-limited (no unit >= 60% of peak; top L1 9.3%) |

Classification rule: highest SOL unit; < 60% of peak for every unit is reported as latency/occupancy-limited. NCU defaults lock clocks to base and flush caches between replays, as in the recorded 5060 Ti capture.


Save results (the `.ncu-rep` files open in the Nsight Compute GUI).

In [6]:
import shutil
archive = shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files
files.download(archive)
# Or keep on Drive:
# from google.colab import drive; drive.mount('/content/drive'); shutil.copy(archive, '/content/drive/MyDrive/')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>